# Day 7 — Decision trees on Titanic: overfitting made visible

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix

## Setup — reuse Day 6's Titanic cleaning

Same feature set as Day 6 (`pclass`, `fare`, `who`, `family_size`). Unlike Day 6, there's no `RobustScaler`/`StandardScaler` step this time — trees split on raw thresholds (`fare <= 26.27`), so scaling changes nothing about where those splits land. That's a real structural difference from logistic regression, not a shortcut.

In [ ]:
df = sns.load_dataset("titanic")
df["sex"] = df["sex"].map({"male": 0, "female": 1})
df["embarked"] = df["embarked"].map({"C": 0, "Q": 1, "S": 2})
df["family_size"] = df["sibsp"] + df["parch"] + 1
df.drop(
    columns=[
        "class",
        "embark_town",
        "alive",
        "alone",
        "sibsp",
        "parch",
        "deck",
        "adult_male",
    ],
    inplace=True,
)

median_age = df["age"].median()
df["age"] = df["age"].fillna(median_age)
mode_embarked = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(mode_embarked)
df["embarked"] = df["embarked"].astype(int)
df["who"] = df["who"].map({"man": 0, "woman": 1, "child": 2})
df.drop(["sex", "age", "embarked"], axis=1, inplace=True)

x_train, x_test, y_train, y_test = train_test_split(
    df.drop("survived", axis=1), df["survived"], test_size=0.2, random_state=42
)

print("train shape:", x_train.shape)
print("test shape: ", x_test.shape)
x_train.describe()

## Step 1 — fit an unconstrained tree, watch it overfit

No `max_depth`, no `min_samples_leaf`, nothing holding it back — `DecisionTreeClassifier(random_state=42)` will keep splitting until every leaf is as pure as it can get. Compare train accuracy against test accuracy, and look at how deep and how wide it actually grew.

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(x_train, y_train)

train_acc = accuracy_score(y_train, tree.predict(x_train))
test_acc = accuracy_score(y_test, tree.predict(x_test))

print(f"train accuracy: {train_acc}")
print(f"test accuracy:  {test_acc}")
print(f"tree depth:  {tree.get_depth()}")
print(f"tree leaves: {tree.get_n_leaves()}")

## Step 2 — the overfitting curve, via cross-validation not raw test accuracy

Step 1 showed one unconstrained tree. Now sweep `max_depth` over `[1, 2, 3, 4, 5, 6, 8, 10, 15, None]` and, at each depth, compute three numbers:

- **train accuracy** — fit on `x_train`, score on `x_train`. Expected to climb with depth (more capacity = better memorization).
- **test accuracy** — fit on `x_train`, score on `x_test`. Touched here only to *watch* it, not to pick a depth from it — with 179 test rows, a couple of flipped predictions move this number by over half a percentage point, so it's a noisy signal on a set this small.
- **CV accuracy** — `cross_val_score(cv=5)`, computed on `x_train` **only**. `x_test` is never passed to it. This is the number that actually decides `max_depth`.

**What 5-fold CV does, mechanically**: split `x_train`'s 712 rows into 5 roughly-equal chunks. For each of the 5 rounds, train a fresh tree on 4 of the chunks (~570 rows) and score it on the 1 held-out chunk (~142 rows) the tree never saw. That gives 5 independent accuracy numbers per depth — report their mean (the estimate) and standard deviation (how much that estimate would wobble on a different random split, i.e. how much you should trust it).

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 15, None]
results = []
for depth in depths:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42)
    t.fit(x_train, y_train)
    train_acc = accuracy_score(y_train, t.predict(x_train))
    test_acc = accuracy_score(y_test, t.predict(x_test))
    cv_scores = cross_val_score(
        DecisionTreeClassifier(max_depth=depth, random_state=42),
        x_train, y_train, cv=5
    )
    results.append((depth, train_acc, test_acc, cv_scores.mean(), cv_scores.std()))
    label = str(depth) if depth is not None else "None"
    print(f"max_depth={label:<4}: train={train_acc:.4f}, test={test_acc:.4f}, CV={cv_scores.mean():.4f} (+/-{cv_scores.std():.4f})")

best_depth, _, _, best_cv, _ = max(results, key=lambda r: r[3])
print(f"\nbest by CV mean: max_depth={best_depth}, CV={best_cv:.4f}")

## Step 3 — one honest evaluation

CV picked `max_depth=3` (Step 2). Retrain a fresh tree at that depth on the full `x_train`, then touch `x_test` exactly once — no more depth-shopping against it. Compare the result to Day 6's L2-regularized logistic regression (0.8100558659217877), which never needed CV because it wasn't searching over a hyperparameter.

In [ ]:
best_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
best_tree.fit(x_train, y_train)

test_preds = best_tree.predict(x_test)
final_test_acc = accuracy_score(y_test, test_preds)
cm = confusion_matrix(y_test, test_preds)

print(f"Final test accuracy at max_depth=3: {final_test_acc}")
print("confusion matrix:\n", cm)

logreg_l2_acc = 0.8100558659217877
print(f"\nDay 6 logistic regression (L2) test accuracy: {logreg_l2_acc}")
print(f"Decision tree (max_depth=3) test accuracy:     {final_test_acc}")
print(f"diff (tree - logreg): {final_test_acc - logreg_l2_acc:+.4f}")

## Step 4 — look inside the tree

Two ways to read a depth-3 tree's learned logic: `.feature_importances_` (a single number per feature — how much total Gini-impurity reduction it's responsible for, summed across every split that used it, normalized to sum to 1), and `plot_tree()` (the actual splits, in order, with thresholds and per-node sample counts). Unlike Day 6's coefficient vector — which tells you *magnitude and direction* but requires you to already understand what "a 1-unit increase in scaled `fare`" means — this is literally readable as a flowchart.

In [ ]:
for name, importance in sorted(
    zip(x_train.columns, best_tree.feature_importances_), key=lambda t: -t[1]
):
    print(f"{name:<12}: {importance:.4f}")

`family_size` never gets used at depth 3 — not because it's useless in principle, but because `who` and `pclass` already explain more of the split-quality gain within the first 3 levels, so the greedy algorithm never reaches for it before running out of depth budget. `who` dominates (0.63) exactly like `sex_male` dominated in the pasted lesson's richer feature set — `who` is built from `sex` (man/woman/child), so this is the same "women and children first" signal your Day 1-2 EDA and Day 6's logistic regression coefficients both already pointed to. Third independent confirmation, from a structurally different model.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    best_tree,
    feature_names=list(x_train.columns),
    class_names=["died", "survived"],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
)
plt.show()